# 07. 통계 검정 (카이제곱, Cramér's V, 비율검정)

**목적**: 01~06번 노트북에서 서술형으로 확인한 H1(도로타입 x 돌발구분), H2(시간대 x 사고
여부), H4(112번/1020번의 예외성)를 카이제곱 독립성 검정과 비율검정으로 통계적으로
검증한다.
**입력 데이터**: `src/load_data.py`의 `load()`가 반환하는 전처리 완료 데이터(623,188행).
**출력물**: 각 가설의 검정통계량/p-value/효과크기를 담은 요약 표(노트북 내 표시, 별도
그림 파일은 생성하지 않는다).
**다음 노트북**: 없음(마지막 노트북). 이 결과는 README 3절 가설 표의 "검정 결과" 열에
반영한다.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from src.load_data import load
from src.notebook_setup import init_notebook

init_notebook()
df = load(with_region=False)
df.shape

(623188, 18)

## 7.1 H1 — 도로타입 × 돌발구분

표본이 **623,188건**으로 매우 크기 때문에, 카이제곱 검정의 p-value는 두 변수 사이에
실제로는 아주 작은 연관성만 있어도 0에 가깝게 나온다. 그래서 "통계적으로 유의한가"보다
**"연관성의 크기(효과크기)가 실질적으로 큰가"**를 Cramér's V로 함께 봐야 한다.

In [2]:
def cramers_v(confusion_matrix: pd.DataFrame) -> float:
    """카이제곱 통계량으로 Cramér's V(0~1, 두 범주형 변수의 연관성 크기)를 계산한다."""
    chi2, p, dof, expected = chi2_contingency(confusion_matrix)
    n = confusion_matrix.values.sum()
    k = min(confusion_matrix.shape) - 1
    v = np.sqrt(chi2 / (n * k))
    return chi2, p, dof, n, v


ct_h1 = pd.crosstab(df['도로타입'], df['돌발구분'])
chi2_h1, p_h1, dof_h1, n_h1, v_h1 = cramers_v(ct_h1)

print(f'chi2 = {chi2_h1:,.2f}')
print(f'dof  = {dof_h1}')
print(f'p    = {p_h1:.3e}')
print(f'n    = {n_h1:,}')
print(f"Cramer's V = {v_h1:.4f}")

chi2 = 159,429.61
dof  = 35
p    = 0.000e+00
n    = 623,188
Cramer's V = 0.2262


**해석**: χ²=159,429.6, df=35, p<0.001로 통계적으로 유의하다. Cramér's V=0.226은
자유도(min(행,열)-1=5) 기준으로 중간 수준의 연관성에 해당해, 03번 노트북에서 확인한
"도로타입마다 지배적 돌발구분이 다르다"는 관찰이 우연이 아니라 실질적인 연관성임을
뒷받침한다.

## 7.2 H2 — 시간대 × 사고 여부

`돌발구분`이 `교통사고`인지 여부를 이분 변수로 두고, 0~23시 각 시간(hour)과의 독립성을
검정한다. H1과 동일하게 표본 크기 때문에 p-value보다 Cramér's V를 기준으로 해석한다.

In [3]:
df['사고여부'] = np.where(df['돌발구분'] == '교통사고', '교통사고', '비사고')
ct_h2 = pd.crosstab(df['hour'], df['사고여부'])
chi2_h2, p_h2, dof_h2, n_h2, v_h2 = cramers_v(ct_h2)

print(f'chi2 = {chi2_h2:,.2f}')
print(f'dof  = {dof_h2}')
print(f'p    = {p_h2:.3e}')
print(f'n    = {n_h2:,}')
print(f"Cramer's V = {v_h2:.4f}")

chi2 = 18,226.93
dof  = 23
p    = 0.000e+00
n    = 623,188
Cramer's V = 0.1710


**해석**: χ²=18,226.9, df=23, p<0.001로 유의하다. 다만 Cramér's V=0.171은
자유도(min(24,2)-1=1) 기준으로는 작은~중간 수준에 그쳐, 시간대가 "사고냐 아니냐"를
가르는 힘은 도로타입만큼 강하지는 않다. 즉 통계적으로는 유의하지만 효과 자체는
크지 않다 — 02번 노트북에서 본 "18시 사고 비중 38.1%" 같은 뚜렷한 피크가 있어도,
사고 발생 자체는 24시간 내내 어느 정도씩 일어나기 때문으로 해석할 수 있다.

## 7.3 H4 — 112번(도시고속국도)·1020번(지방도)의 예외성

05번 노트북에서 "도로타입 평균과 크게 다른 노선"으로 지목한 112번·1020번이 실제로
같은 도로타입의 나머지 노선과 통계적으로 다른 비중을 보이는지 two-proportion z-test로
확인한다. README의 "도로타입 평균"에는 112번/1020번 자신도 포함되어 있어, 검정에서는
해당 노선을 제외한 "나머지 노선"을 비교군으로 쓴다(그래서 아래 비교 비율이 README의
평균값과 소수점에서 약간 다르다).

In [4]:
sub_urban = df[df['도로타입'] == '도시고속국도'].copy()
sub_urban['도로번호'] = sub_urban['도로번호'].astype('Int64')
is_112 = sub_urban['도로번호'] == 112

n1, x1 = int(is_112.sum()), int((sub_urban.loc[is_112, '돌발구분'] == '공사').sum())
n2, x2 = int((~is_112).sum()), int((sub_urban.loc[~is_112, '돌발구분'] == '공사').sum())

z_112, p_112 = proportions_ztest([x1, x2], [n1, n2])

print(f'112번 공사 비중     = {x1}/{n1} = {x1/n1:.1%}')
print(f'나머지 노선 공사 비중 = {x2}/{n2} = {x2/n2:.1%}')
print(f'z = {z_112:.3f}, p = {p_112:.3e}')

112번 공사 비중     = 63/74 = 85.1%
나머지 노선 공사 비중 = 5299/23524 = 22.5%
z = 12.833, p = 1.075e-37


In [5]:
sub_local = df[df['도로타입'] == '지방도'].copy()
sub_local['도로번호'] = sub_local['도로번호'].astype('Int64')
is_1020 = sub_local['도로번호'] == 1020

n3, x3 = int(is_1020.sum()), int((sub_local.loc[is_1020, '돌발구분'] == '교통사고').sum())
n4, x4 = int((~is_1020).sum()), int((sub_local.loc[~is_1020, '돌발구분'] == '교통사고').sum())

z_1020, p_1020 = proportions_ztest([x3, x4], [n3, n4])

print(f'1020번 교통사고 비중     = {x3}/{n3} = {x3/n3:.1%}')
print(f'나머지 노선 교통사고 비중 = {x4}/{n4} = {x4/n4:.1%}')
print(f'z = {z_1020:.3f}, p = {p_1020:.3e}')

1020번 교통사고 비중     = 33/45 = 73.3%
나머지 노선 교통사고 비중 = 993/5138 = 19.3%
z = 9.053, p = 1.395e-19


**해석**: 112번(85.1% vs 나머지 22.5%, z=12.83, p<0.001)과 1020번(73.3% vs 나머지
19.3%, z=9.05, p<0.001) 모두 같은 도로타입의 나머지 노선과 비교해 통계적으로 유의하게
다르다. 표본 수가 112번(74건)·1020번(45건)으로 크지 않은데도 비율 차이 자체가
워낙 커서(약 3~4배) 유의한 결과가 나왔다 — H1·H2와 달리 이 경우는 표본 크기 효과가
아니라 실제 비율 차이가 크다는 뜻이다.

## 7.4 요약

In [6]:
summary = pd.DataFrame([
    {'가설': 'H1 도로타입 x 돌발구분', '검정': '카이제곱 독립성 검정',
     '통계량': f'chi2={chi2_h1:,.1f}, df={dof_h1}', 'p-value': f'{p_h1:.1e}',
     '효과크기': f"Cramer's V={v_h1:.3f}"},
    {'가설': 'H2 시간대 x 사고여부', '검정': '카이제곱 독립성 검정',
     '통계량': f'chi2={chi2_h2:,.1f}, df={dof_h2}', 'p-value': f'{p_h2:.1e}',
     '효과크기': f"Cramer's V={v_h2:.3f}"},
    {'가설': 'H4 112번 vs 나머지(도시고속국도)', '검정': 'two-proportion z-test',
     '통계량': f'z={z_112:.2f}', 'p-value': f'{p_112:.1e}',
     '효과크기': f'{x1/n1:.1%} vs {x2/n2:.1%}'},
    {'가설': 'H4 1020번 vs 나머지(지방도)', '검정': 'two-proportion z-test',
     '통계량': f'z={z_1020:.2f}', 'p-value': f'{p_1020:.1e}',
     '효과크기': f'{x3/n3:.1%} vs {x4/n4:.1%}'},
])
summary

,가설,검정,통계량,p-value,효과크기
0,H1 도로타입 x 돌발구분,카이제곱 독립성 검정,"chi2=159,429.6, df=35",0.0e+00,Cramer's V=0.226
1,H2 시간대 x 사고여부,카이제곱 독립성 검정,"chi2=18,226.9, df=23",0.0e+00,Cramer's V=0.171
2,H4 112번 vs 나머지(도시고속국도),two-proportion z-test,z=12.83,1.1e-37,85.1% vs 22.5%
3,H4 1020번 vs 나머지(지방도),two-proportion z-test,z=9.05,1.4e-19,73.3% vs 19.3%
